In [1]:
import sys, pathlib as pl; sys.path.insert(0, str(pl.Path.cwd().parents[1])); from fig_utils import fig_csv
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import uncertainty_toolbox as uct

from utils import savefig

In [2]:
import pathlib as pl
import candas as can

code_pth = pl.Path.cwd()  # for running in Jupyter
# code_pth = pl.Path(__file__)  # for running in terminal
fig_pth = code_pth.parent
data_pth = fig_pth / "data"
graph_pth = fig_pth / "graphics"
graph_pth.mkdir(exist_ok=True)

gen_pth = fig_pth / "generated"
gen_pth.mkdir(exist_ok=True)

plt.style.use(str(can.style.breve))



In [3]:
ticklabelsize = 6
labelsize = 8
titlesize = labelsize + 2

plt.rcParams["lines.linewidth"] = 1
plt.rcParams["axes.linewidth"] = 0.5
plt.rcParams["grid.linewidth"] = 0.25
plt.rcParams["font.size"] = labelsize
plt.rcParams["axes.titlesize"] = titlesize
plt.rcParams["axes.labelsize"] = labelsize
plt.rcParams["xtick.labelsize"] = ticklabelsize
plt.rcParams["ytick.labelsize"] = ticklabelsize
plt.rcParams["xtick.major.size"] = 1.5
plt.rcParams["ytick.major.size"] = 1.5
plt.rcParams["xtick.major.width"] = 0.5
plt.rcParams["ytick.major.width"] = 0.5
plt.rcParams["legend.fontsize"] = labelsize
plt.rcParams["legend.title_fontsize"] = labelsize
plt.rcParams["figure.titlesize"] = titlesize

In [4]:
qq_avg = pd.read_csv(gen_pth / "Avg_model_qq.csv")
qq_ind = pd.read_csv(gen_pth / "Ind_model_qq.csv")
qq_lmc_pr = pd.read_csv(gen_pth / "LMC_model_qq.csv")
qq = pd.concat([qq_avg.assign(
    Model="Average",
), qq_ind.assign(
    Model="Individual",
), qq_lmc_pr.assign(
    Model="LMC_pr",
)], axis=0)

if (gen_pth / "LMC_model2_qq.csv").exists():
    qq_lmc_p = pd.read_csv(gen_pth / "LMC_model2_qq.csv")
    qq = pd.concat([qq, qq_lmc_p.assign(
        Model="LMC_p",
    )], axis=0)
    
qq = qq.rename(columns={
    "N_train": "N Training Points",
    'Train_pred_q': 'Expected Proportion',
    'Train_obsd_q': 'Observed Proportion',
}).reset_index(drop=True)

In [9]:
Models = qq["Model"].unique()
n_m = len(Models)
print(n_m)
fig, axs = plt.subplots(1, n_m, figsize=(7, 2.5), sharex=True, sharey=True)

for _i_ax, (ax, Model) in enumerate(zip(axs.flat, Models)):

    miscals = []
    data = qq[qq["Model"] == Model].reset_index(drop=True)
    for i in data.Iteration.unique():
        data_i = data[data["Iteration"] == i]
        ax.plot(
            data_i["Expected Proportion"],
            data_i["Observed Proportion"],
            color="C0",
            alpha=0.5,
            linewidth=0.5,
        )
        print(data_i["Expected Proportion"][0:50])
        print(data_i["Observed Proportion"][0:50])

        miscals.append(
            uct.metrics_calibration.miscalibration_area_from_proportions(
                exp_proportions=data_i["Expected Proportion"].values,
                obs_proportions=data_i["Observed Proportion"].values,
            )
        )

    grpd_obs = data.groupby("Expected Proportion")["Observed Proportion"]
    l, m, u = [grpd_obs.quantile(q) for q in [0.05, 0.5, 0.95]]
    ax.plot(
        data_i["Expected Proportion"],
        m,
        color="C0",
        linewidth=2,
        label="Median",
    )
    ax.fill_between(
        data_i["Expected Proportion"],
        l,
        u,
        color="C0",
        alpha=0.2,
        label="5-95% quantiles",
        edgecolor="none",
    )

    #print(data_i["Expected Proportion"])
    #print(m)
    #print(u)
    #print(l)
    data_i["Expected Proportion"].to_csv(fig_csv('wet_iguana', suffix=f'qq{_i_ax}'))
    ax.plot([0, 1], [0, 1], color="k", linestyle="--", linewidth=0.5)
    
    l_of_mis, m_of_mis, u_of_mis = np.quantile(np.stack(miscals), [0.05, 0.5, 0.95], axis=-1)
    mis_of_l, mis_of_m, mis_of_u = [uct.metrics_calibration.miscalibration_area_from_proportions(
        exp_proportions=data_i["Expected Proportion"].values,
        obs_proportions=mis.values,
    ) for mis in [l, m, u]]


    model = Model.replace("_pr", "$_{pr}$").replace("_p", "$_{p}$")
    ax.set_title(f"{model}\n"
                 f"RMA: {m_of_mis:.3f} ({l_of_mis:.3f}, {u_of_mis:.3f})\n"
                 f"SMA: {mis_of_m:.3f}",
                 fontsize=titlesize)
    ax.grid(True)
    ax.set_ylim(0, 1)
    ax.set_xlim(0, 1)
    ax.set_aspect("equal")
    ax.set_xlabel("Expected Proportion")
    
axs[0].set_ylabel("Observed Proportion")

plt.tight_layout()
#savefig(plt.gcf(), alias="wet_iguana")

4
0     0.000000
1     0.010101
2     0.020202
3     0.030303
4     0.040404
5     0.050505
6     0.060606
7     0.070707
8     0.080808
9     0.090909
10    0.101010
11    0.111111
12    0.121212
13    0.131313
14    0.141414
15    0.151515
16    0.161616
17    0.171717
18    0.181818
19    0.191919
20    0.202020
21    0.212121
22    0.222222
23    0.232323
24    0.242424
25    0.252525
26    0.262626
27    0.272727
28    0.282828
29    0.292929
30    0.303030
31    0.313131
32    0.323232
33    0.333333
34    0.343434
35    0.353535
36    0.363636
37    0.373737
38    0.383838
39    0.393939
40    0.404040
41    0.414141
42    0.424242
43    0.434343
44    0.444444
45    0.454545
46    0.464646
47    0.474747
48    0.484848
49    0.494949
Name: Expected Proportion, dtype: float64
0     0.000000
1     0.000000
2     0.000000
3     0.000000
4     0.000000
5     0.032258
6     0.032258
7     0.032258
8     0.096774
9     0.096774
10    0.129032
11    0.129032
12    0.129032
13    0.129